**Import Required Libraries**

In [1]:
!pip install -U langchain langchain-community sentence-transformers faiss-cpu rank_bm25 transformers accelerate pypdf unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━

In [2]:
import torch
print(torch.cuda.is_available())

False


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.llms import Ollama

from sentence_transformers import CrossEncoder
from transformers import pipeline

**Ollama Setup**

In [4]:
!apt-get update -qq
!apt-get install -y zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 118 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (907 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:

In [5]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [6]:
!ollama pull llama3:8b
!ollama pull mxbai-embed-large
!ollama list



NAME                        ID              SIZE      MODIFIED               
mxbai-embed-large:latest    468836162de7    669 MB    Less than a second ago    
llama3:8b                   365c0bd3c000    4.7 GB    12 seconds ago            


**Load PDF Document**

In [7]:
loader = PyPDFLoader("/content/2 REFRIGERANT MANAGEMENT PLAN.pdf")
pages = loader.load()

**Extract Text from PDF Pages**

In [8]:
page_texts = []

for page in pages:
    page_texts.append(page.page_content)

print("Text pages extracted:", len(page_texts))

Text pages extracted: 14


CLEAN PDF TEXT (REMOVE HEADERS, FOOTERS, PAGE NOISE)

In [9]:
def clean_text(text):

    # Remove page numbers
    text = re.sub(r'Page\s*\d+', '', text)

    # Remove footer line
    text = re.sub(r'Printed by Hanwha Ocean[^\n]*', '', text)

    # Remove Adobe notice
    text = re.sub(r'Document must be opened with Adobe to view all ABS amendments\.', '', text)

    # Remove repeated headers
    text = re.sub(r'Refrigerant Management Plan', '', text)

    # Remove timestamps
    text = re.sub(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}', '', text)

    # Remove Adobe notice multiline
    text = re.sub(r'Document must.*?amendments\.', '', text, flags=re.DOTALL)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Fix broken lines
    text = text.replace('\n', ' ')

    # Strip spaces
    text = text.strip()

    return text

In [10]:
import re

In [11]:
clean_pages = []

for page in page_texts:
    cleaned = clean_text(page)
    clean_pages.append(cleaned)

print("Total cleaned pages:", len(clean_pages))

Total cleaned pages: 14


Create LangChain Documents from Cleaned PDF Pages

In [12]:
from langchain_core.documents import Document

documents = []

for i, text in enumerate(clean_pages):

    doc = Document(
        page_content=text,
        metadata={"page": i + 1}
    )

    documents.append(doc)

print("Documents created:", len(documents))

Documents created: 14


Chunk the Document

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]

)
docs = text_splitter.split_documents(documents)

Create Embeddings

In [14]:
embedding_model = OllamaEmbeddings(
        model="mxbai-embed-large"
    )

/tmp/ipykernel_422/3040894951.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding_model = OllamaEmbeddings(


Create Vector Database (FAISS)

In [15]:
vector_db = FAISS.from_documents(docs, embedding_model)

vector_retriever = vector_db.as_retriever(
    search_kwargs={"k":7}
)

Create BM25 Retriever

In [16]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 7

H**ybrid Retrieval (Vector + BM25)**

In [17]:
def hybrid_retrieve(query):

    vec_docs = vector_retriever.invoke(query)
    bm_docs = bm25_retriever.invoke(query)

    docs = vec_docs + bm_docs

    # remove duplicates
    unique = {d.page_content: d for d in docs}

    return list(unique.values())

**Cross-Encoder Reranker**

In [18]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=2):

    pairs = [(query, d.page_content) for d in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for _, doc in ranked[:top_k]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LLM (Ollama)

In [19]:
from langchain_community.llms import Ollama

qa_model = Ollama(
    model="llama3:8b",
    temperature=0
)

/tmp/ipykernel_422/2876701148.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  qa_model = Ollama(


Structured Prompt

In [20]:
def build_prompt(context, question):

    prompt = f"""
You are an expert assistant answering questions from a ship manual.

Use ONLY the provided context to answer.

If the answer is not present, say:
"Answer not found in document."

Context:
{context}

Question:
{question}

Answer clearly and concisely.
Also mention the page numbers if available.
"""

    return prompt

Final ask_question() Function

In [21]:
def ask_question(query):

    # Step 1: Hybrid Retrieval
    docs = hybrid_retrieve(query)

    # Step 2: Rerank
    docs = rerank(query, docs)

    # Step 3: Prepare Context
    context = "\n\n".join(
        f"[Page {d.metadata.get('page', 'N/A')}]\n{d.page_content}"
        for d in docs
    )

    # Step 4: Prompt
    prompt = build_prompt(context, query)

    # Step 5: LLM
    response = qa_model.invoke(prompt)

    return response

In [22]:
query =  "What is the purpose of the Refrigerant Management Plan?"


answer = ask_question(query)
print("question\n",query )

print("answer\n",answer)

question
 What is the purpose of the Refrigerant Management Plan?
answer
 According to the provided context, the purpose of the Refrigerant Management Plan is:

"To provide as a source of information and guidance which will enable a historic and accurate record of refrigerant leakage detection/disposal."

This is stated on Page 5, Section 1.1.


**RAG System Performance Evaluation Report**

Total Questions: 31

Accuracy Distribution

100% → 24 questions

95% → 3 questions

90% → 3 questions

85% → 1 question

Evaluation results (31 questions, 97.26% accuracy)

**Tools and Libraries Used**

 Framework       | LangChain           
 LLM             | Ollama (llama3:8b)  
 Vector Database | FAISS               
 Embedding Model | ollama(mxbai-embed-large )

Data File Link:
https://drive.google.com/file/d/1Y_6Q7yZF4Se6AxyEG20gqMMRF_hu0m9m/view?usp=drive_link